# Analisis MGWR dengan Dataset Baru

Notebook ini menjalankan analisis Multiscale Geographically Weighted Regression (MGWR) untuk melihat apakah setiap faktor pendorong NEET beroperasi pada skala geografis yang berbeda.

## 1. Persiapan Pustaka dan Data

Memuat pustaka, data sosio-ekonomi baru, dan data koordinat geografis. Data ini akan digabungkan dan dibersihkan untuk persiapan model.

In [14]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from mgwr.gwr import GWR, MGWR
from mgwr.sel_bw import Sel_BW

# --- Ganti path ini dengan lokasi file Anda ---
file_path = 'Data/DATASET_FINAL_PROVINSI.csv' 
df = pd.read_csv(file_path)

# Hapus baris data agregat jika ada
if 'INDONESIA' in df['Provinsi'].values:
    df = df[df['Provinsi'] != 'INDONESIA'].copy()

# --- Data Koordinat (Lintang & Bujur) ---
coords_data = {
    'Provinsi': ['ACEH', 'SUMATERA UTARA', 'SUMATERA BARAT', 'RIAU', 'JAMBI', 'SUMATERA SELATAN', 'BENGKULU', 'LAMPUNG', 'KEP. BANGKA BELITUNG', 'KEP. RIAU',
                 'DKI JAKARTA', 'JAWA BARAT', 'JAWA TENGAH', 'DI YOGYAKARTA', 'JAWA TIMUR', 'BANTEN',
                 'BALI', 'NUSA TENGGARA BARAT', 'NUSA TENGGARA TIMUR',
                 'KALIMANTAN BARAT', 'KALIMANTAN TENGAH', 'KALIMANTAN SELATAN', 'KALIMANTAN TIMUR', 'KALIMANTAN UTARA',
                 'SULAWESI UTARA', 'SULAWESI TENGAH', 'SULAWESI SELATAN', 'SULAWESI TENGGARA', 'GORONTALO', 'SULAWESI BARAT',
                 'MALUKU', 'MALUKU UTARA', 'PAPUA BARAT', 'PAPUA BARAT DAYA', 'PAPUA', 'PAPUA SELATAN', 'PAPUA TENGAH', 'PAPUA PEGUNUNGAN'],
    'Latitude': [-4.36, -3.58, -0.92, 0.51, -1.61, -2.99, -3.79, -5.45, -2.71, 1.13,
                 -6.18, -6.92, -7.15, -7.79, -7.26, -6.45,
                 -8.33, -8.58, -8.65,
                 -0.02, -1.50, -3.32, 1.25, 2.98,
                 1.47, -1.69, -5.16, -3.98, 0.54, -2.67,
                 -3.29, 0.79, -0.87, -0.87, -2.53, -6.00, -4.00, -4.00],
    'Longitude': [97.02, 98.67, 100.37, 101.45, 103.61, 104.76, 102.26, 105.27, 106.36, 104.05,
                  106.83, 107.62, 110.42, 110.37, 112.75, 106.14,
                  115.09, 116.12, 123.58,
                  109.34, 113.29, 114.59, 117.15, 117.38,
                  124.84, 120.81, 119.43, 122.52, 123.06, 118.89,
                  130.18, 127.36, 134.08, 131.25, 140.72, 140.38, 136.00, 139.00]
}
df_coords = pd.DataFrame(coords_data)

# --- Menyamakan format nama provinsi untuk penggabungan ---
df['Provinsi_Upper'] = df['Provinsi'].str.upper()
df_coords['Provinsi_Upper'] = df_coords['Provinsi'].str.upper()

# --- Gabungkan DataFrame ---
mgwr_df = pd.merge(df, df_coords, on='Provinsi_Upper', how='left')
mgwr_df.drop(columns=['Provinsi_Upper', 'Provinsi_y'], inplace=True)
mgwr_df.rename(columns={'Provinsi_x': 'Provinsi'}, inplace=True)

# --- Persiapan Variabel ---
y_var = 'NEET_15-24_persen'
x_vars = ['Rata_Rata_Lama_Sekolah', 'PDRB_per_Kapita', 'Tingkat_Kemiskinan_persen']
model_df = mgwr_df[[y_var] + x_vars + ['Latitude', 'Longitude']].dropna()

y = model_df[y_var].values.reshape(-1, 1)
X = model_df[x_vars].values
coords = list(zip(model_df['Longitude'], model_df['Latitude']))
X_scaled = StandardScaler().fit_transform(X)

print(f"Data siap untuk MGWR dengan {len(model_df)} observasi.")

Data siap untuk MGWR dengan 38 observasi.


## 2. Pencarian Bandwidth Multiskala dan Pelatihan Model

**Peringatan:** Sel di bawah ini akan berjalan cukup lama (bisa beberapa menit) karena proses komputasi untuk mencari bandwidth optimal bagi setiap variabel secara terpisah.

In [18]:
print("--- Memulai pencarian bandwidth multiskala (ini akan memakan waktu)... ---")
n_vars = X_scaled.shape[1] + 1  # jumlah variabel + intercept
selector = Sel_BW(coords, y, X_scaled, multi=True, fixed=True)
selector.search(
    criterion='AICc',
    tol=1.e-5,
    max_iter=200,
    multi_bw_min=[2]*n_vars,
    multi_bw_max=[len(model_df)//2]*n_vars,
    bw_min=2,
    bw_max=len(model_df)//2
)

# Validasi dan batasi bandwidth hasil pencarian (flatten jika nested)
def flatten_bw(bw):
    flat = []
    for b in bw:
        if isinstance(b, (list, np.ndarray)):
            # Jika array/list kosong, skip
            if len(b) == 0:
                continue
            # Ambil elemen pertama yang bisa dikonversi ke float
            found = False
            for item in b:
                try:
                    flat.append(float(item))
                    found = True
                    break
                except Exception:
                    continue
            if not found:
                continue
        else:
            try:
                flat.append(float(b))
            except Exception:
                continue
    return flat
# ...existing code...

raw_bw = flatten_bw(selector.bw)
mgwr_bws = [min(int(b), len(model_df)-1) for b in raw_bw]
print(f"Bandwidth MGWR yang digunakan (untuk Intercept, RLS, PDRB, Kemiskinan): {mgwr_bws}")

# Fungsi untuk memastikan semua elemen adalah integer (tidak nested sama sekali dan TIDAK ada list di dalam list)
def force_flat_int_list(lst):
    out = []
    for x in lst:
        # Jika x adalah list/array, ambil hanya elemen pertamanya (dan pastikan bukan list lagi)
        if isinstance(x, (list, np.ndarray)):
            # Jika elemen pertama juga list, rekursif
            if len(x) > 0:
                if isinstance(x[0], (list, np.ndarray)):
                    out.extend(force_flat_int_list(x[0]))
                else:
                    out.append(int(x[0]))
        else:
            out.append(int(x))
    return out

mgwr_bws_flat = force_flat_int_list(mgwr_bws)
print(f"Bandwidth final yang dipakai MGWR: {mgwr_bws_flat}")
print(f"Tipe setiap elemen: {[type(x) for x in mgwr_bws_flat]}")  # Debug

# Pastikan hasil akhirnya list of int, bukan list of list
assert all(isinstance(x, int) for x in mgwr_bws_flat), f"Bandwidth masih nested: {mgwr_bws_flat}"

selector.bw = mgwr_bws_flat
if hasattr(selector, "bw_init"):
    selector.bw_init = mgwr_bws_flat
if hasattr(selector, "bws"):
    selector.bws = mgwr_bws_flat

print("\n--- Melatih model MGWR... ---")
mgwr_model = MGWR(coords, y, X_scaled, selector=selector)
mgwr_results = mgwr_model.fit()

print("\n--- Ringkasan Model MGWR ---")
print(mgwr_results.summary())

--- Memulai pencarian bandwidth multiskala (ini akan memakan waktu)... ---


Backfitting:   0%|          | 0/200 [00:00<?, ?it/s]

Bandwidth MGWR yang digunakan (untuk Intercept, RLS, PDRB, Kemiskinan): [8, 0, 4, 37]
Bandwidth final yang dipakai MGWR: [8, 0, 4, 37]
Tipe setiap elemen: [<class 'int'>, <class 'int'>, <class 'int'>, <class 'int'>]

--- Melatih model MGWR... ---


C:\Users\LENOVO\AppData\Local\Temp\ipykernel_25400\2137039955.py:26: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  flat.append(float(item))


TypeError: int() argument must be a string, a bytes-like object or a real number, not 'list'

## 3. Interpretasi Hasil

Setelah model selesai, perhatikan beberapa hal kunci di ringkasan output di atas:

1.  **Bandwidths:** Lihat nilai bandwidth yang berbeda untuk setiap variabel (`X1`, `X2`, `X3`).
    * **Bandwidth Kecil:** Menandakan pengaruh variabel tersebut sangat **lokal** (hanya dipengaruhi beberapa tetangga terdekat).
    * **Bandwidth Besar:** Menandakan pengaruh variabel tersebut lebih **global** atau regional (dipengaruhi oleh banyak provinsi lain).
2.  **AICc:** Bandingkan nilai AICc dari model MGWR ini dengan model GWR sebelumnya. Jika AICc MGWR lebih rendah, maka secara statistik model ini dianggap lebih baik dalam menjelaskan data.
3.  **R-squared:** Lihat apakah ada peningkatan pada nilai R-squared dibandingkan model GWR sebelumnya.